# Image registration

> description: Rotate image based on reference image



In [ ]:
#| default_exp preprocessing.image_registration

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from pathlib import Path
from fastcore.all import *
import numpy as np
import os
import sys
import yaml
import cv2
from fastcore.imports import *
from PIL import Image
from dataclasses import dataclass
from skimage.registration import phase_cross_correlation
from typing import List, Tuple

In [ ]:
#| export
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import math


In [ ]:
#| export
@dataclass
class ETImageRegistration:
    registered_test_et: np.ndarray
    rotation_deg: float
    translation_xy: tuple[float, float]
    transformation_matrix: np.ndarray # 2x3 affine s
    dice_after: float
    registration_ok: bool
    

In [ ]:
#| export
def to_gray(img: np.ndarray) -> np.ndarray:
    """Convert image to grayscale if it's not already."""
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3  else img

In [ ]:
TYPE_C_150='1C_Typ150'
DIRECTION='Incoming'
DATA_PATH =Path(r'/home/ai_dsx.work/data/projects/goni/hochstrom_pin_ets')
IM_PATH_1c150 = Path(DATA_PATH,TYPE_C_150,DIRECTION, f'{DIRECTION}_unzip', 'main_im2')
DATA_PATH.ls().map(lambda x: print(x.name));


In [ ]:
ref_im_path = Path(DATA_PATH, 'REFERENCE_IMAGE_EVAL')
ref_im_path.ls().map(lambda x: print(x.name))

In [ ]:
r_1c = Path(ref_im_path, 'VFV4.9.0.3_2025100513085192_ID_33975099923900001901111_In_150_FRONT_Pass_image2_1C.png')
r_2c = Path(ref_im_path, 'VFV4.9.0.3_2025101902465665_ID_00001053758820214772541_In_149_FRONT_Flying Lead_image2_2C.png')
r_1c.exists(), r_2c.exists()

In [ ]:
#| export
from cv_tools.core import *
from cv_tools.image_alignment import *

In [ ]:
reference_image = center_crop(Image.fromarray(cv2.imread(r_1c, cv2.IMREAD_GRAYSCALE))) 
sm_1c_150 = np.random.choice(IM_PATH_1c150.ls(),1)[0]
tst_im = center_crop(Image.fromarray(cv2.imread(sm_1c_150, cv2.IMREAD_GRAYSCALE)))


In [ ]:
warp_matrix = np.eye(2,3, dtype=np.float32)
warp_matrix

In [ ]:
crit = (
    cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
    1000, # max iterations
    1e-6 # convergence threshold -tight for pin-level accuracy
)

In [ ]:
print(f'Step 1: ECC registration (Euclidean:rotation + translation)..')
cc, warp_matrix = cv2.findTransformECC(
    to_gray(reference_image),
    to_gray(tst_im),
    warp_matrix,
    cv2.MOTION_EUCLIDEAN,
    crit,
    None,  # inputMask parameter
    5      # gaussFiltSize parameter
)
print(f'Ecc correlation: {cc:.6f}')
rot_deg = np.degrees(np.arctan2(warp_matrix[1, 0], warp_matrix[0, 0]))
translation_xy = (warp_matrix[0,2], warp_matrix[1,2]) # (dx, dy)
print(f'Rotation: {rot_deg:.2f} degrees')
print(f'Translation dx: {translation_xy[0]:.3f}px, dy= {translation_xy[1]:.3f}px')



In [ ]:
rg = cv2.warpAffine(
    tst_im,
    warp_matrix,
    (tst_im.shape[1], tst_im.shape[0]),
    flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP,
    borderMode=cv2.BORDER_REPLICATE 
    )
    

In [ ]:
# NCC = Normalized Cross-Correlation
# This measures how similar two images are after registration
# NCC ranges from -1 to 1, where:
#   1.0 = perfect match (images are identical)
#   0.0 = no correlation 
#  -1.0 = perfect inverse match (one is negative of the other)

# Step 1: Normalize both images to have values between 0-1
ref_norm = reference_image.astype(np.float32) / (reference_image.max()  + 1e-8)
tst_norm = rg.astype(np.float32) / (rg.max()  + 1e-8)

# Step 2: Calculate NCC using the formula:
# NCC = (sum of pixel-wise products) / (product of image magnitudes)
# This is essentially the cosine of the angle between two image vectors
ncc = float(
    np.sum(ref_norm * tst_norm) / (
        np.sqrt(np.sum(ref_norm**2)) * np.sqrt(np.sum(tst_norm**2)) + 1e-8
    )
)
print(f'NCC: {ncc:.4f} (1.0=perfect match, 0.0=no correlation)')


In [ ]:
display_imgs(images=[reference_image, rg], scale=1, COLS=2, desc=None);

In [ ]:
#| export
from loguru import logger

In [ ]:
#| export
def fallback_alignment(
    ref_gray: np.ndarray,
    tst_gray: np.ndarray,
    angle_limit: float,
    upsample_factor: int
    ) -> tuple[np.ndarray, float, tuple[float, float]]:
    """Fallback alignment using phase correlation + Fourier-Mellin."""

    # Helper function to convert image to log-polar FFT representation
    # This transforms rotation into translation for easier detection
    def log_polar_fft(img):
        # Apply Hanning window to reduce edge artifacts in FFT
        # Outer product creates 2D window from 1D Hanning windows
        windowed_img = img * np.outer(
            np.hanning(img.shape[0]),  # Vertical window
            np.hanning(img.shape[1])   # Horizontal window
        )
        
        # Compute 2D FFT and shift zero frequency to center
        fft_result = np.fft.fftshift(np.fft.fft2(windowed_img))
        
        # Take absolute value to get magnitude spectrum
        magnitude = np.abs(fft_result)
        
        # Apply log transform to compress dynamic range (log1p = log(1+x))
        return np.log1p(magnitude)

    # Get image dimensions for coordinate calculations
    h, w = ref_gray.shape

    # Define center point for polar transformation
    center = (w//2, h//2)
    
    # Set flags for polar warping: logarithmic + fill empty areas
    flags = cv2.WARP_POLAR_LOG + cv2.WARP_FILL_OUTLIERS

    # Transform reference image to log-polar coordinates
    # This converts rotations to horizontal shifts
    ref_polar = cv2.warpPolar(
        log_polar_fft(ref_gray),  # Input: log-polar FFT of reference
        (360, 360),               # Output size: 360x360 pixels
        center,                   # Center point for transformation
        min(center) * 0.8,        # Radius (80% of min dimension)
        flags                     # Transformation flags
    )

    # Transform test image to log-polar coordinates using same parameters
    tst_polar = cv2.warpPolar(
        log_polar_fft(tst_gray),  # Input: log-polar FFT of test image
        (360, 360),               # Same output size as reference
        center,                   # Same center point
        min(center) * 0.8,        # Same radius
        flags                     # Same transformation flags
    )

    # Find shift between polar images using phase correlation
    # This detects rotation (as horizontal shift in polar space)
    shift, _, _ = phase_cross_correlation(
        ref_polar,                # Reference polar image
        tst_polar,                # Test polar image
        upsample_factor=upsample_factor  # Subpixel accuracy factor
    )

    # Convert horizontal shift in polar space back to rotation angle
    # shift[0] is horizontal shift, convert to degrees
    rot_deg = float(shift[0] * 360.0 / 360.0)

    # Handle angle wrapping: if rotation > limit, try opposite direction
    # This handles 180-degree ambiguity in rotation detection
    if abs(rot_deg) > angle_limit:
        rot_deg -= np.sign(rot_deg) * 180.0

    # Create rotation matrix around image center
    M_rot = cv2.getRotationMatrix2D(center, rot_deg, 1.0)

    # Apply rotation to test image to correct for detected rotation
    test_rotated = cv2.warpAffine(
        tst_gray,                 # Input test image
        M_rot,                    # Rotation transformation matrix
        (w, h),                   # Output size (same as input)
        flags=cv2.INTER_LINEAR,   # Linear interpolation
    )

    # Now find translation between reference and rotated test image
    # This detects remaining translation after rotation correction
    shift_t_, _, _ = phase_cross_correlation(
        ref_gray.astype(np.float32),      # Reference as float32
        test_rotated.astype(np.float32),  # Rotated test as float32
        upsample_factor=upsample_factor   # Subpixel accuracy
    )

    # Extract translation components (dx, dy)
    dx, dy = float(shift_t_[0]), float(shift_t_[1])

    # Build combined affine transformation matrix
    # This combines rotation and translation into single matrix
    angle_rad = np.deg2rad(-rot_deg)  # Convert to radians, negate for correct direction
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)  # Trigonometric values
    cx, cy = center  # Center coordinates

    # Create 2x3 affine matrix: [rotation + translation]
    # Formula accounts for rotation around center point + final translation
    warp_matrix = np.float32([
        [cos_a, -sin_a, cx + dx - cx * cos_a + cy * sin_a],  # Row 1: x transformation
        [sin_a, cos_a, cy + dy - cx * sin_a - cy * cos_a]    # Row 2: y transformation
    ])

    # Return transformation matrix, rotation angle, and translation
    return warp_matrix, rot_deg, (dx, dy)


In [ ]:
#| export
def register_image(
    ref_im: np.ndarray,
    tst_im: np.ndarray,
    angle_limit: float = 6.0, # degrees
    upsample_factor: int = 20, # for subpixel accuracy
    use_fallback: bool = False, # kept for API compatibility, ignored - failures always return tst_im unchanged
    ) -> ETImageRegistration:
    """Register tst_im to ref_im using ECC.
    
    If ECC fails to converge OR rotation exceeds angle_limit:
        returns tst_im unchanged with registration_ok=False.
    Always returns ETImageRegistration with the same output format.
    """

    def _to_gray(img: np.ndarray) -> np.ndarray:
        """Convert image to grayscale if it's not already."""
        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img

    def _compute_ncc(ref_gray, tst_gray):
        # Resize tst to match ref if shapes differ
        if ref_gray.shape != tst_gray.shape:
            tst_gray = cv2.resize(tst_gray, (ref_gray.shape[1], ref_gray.shape[0]))
        ref_norm = ref_gray.astype(np.float32) / (ref_gray.max() + 1e-8)
        tst_norm = tst_gray.astype(np.float32) / (tst_gray.max() + 1e-8)
        return float(
            np.sum(ref_norm * tst_norm) / (
                np.sqrt(np.sum(ref_norm**2)) * np.sqrt(np.sum(tst_norm**2)) + 1e-8
            )
        )

    def _no_registration_result(reason: str) -> ETImageRegistration:
        """Return tst_im unchanged with registration_ok=False."""
        logger.warning(reason)
        ncc = _compute_ncc(ref_gray, tst_gray)
        logger.info(f'NCC: {ncc:.4f} (no registration applied)')
        return ETImageRegistration(
            registered_test_et=tst_im,
            rotation_deg=0.0,
            translation_xy=(0.0, 0.0),
            transformation_matrix=np.eye(2, 3, dtype=np.float32),
            dice_after=ncc,
            registration_ok=False
        )

    ref_gray = _to_gray(ref_im)
    tst_gray = _to_gray(tst_im)

    warp_matrix = np.eye(2, 3, dtype=np.float32)

    crit = (
        cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
        1000, # max iterations
        1e-6  # convergence threshold
    )

    # Resize tst_gray to match ref_gray for ECC if shapes differ
    tst_gray_ecc = tst_gray
    if ref_gray.shape != tst_gray.shape:
        logger.warning(f'Image shapes differ: ref={ref_gray.shape}, tst={tst_gray.shape} - resizing tst for ECC')
        tst_gray_ecc = cv2.resize(tst_gray, (ref_gray.shape[1], ref_gray.shape[0]))

    logger.info('Step 1: ECC registration (Euclidean: rotation + translation)..')
    try:
        cc, warp_matrix = cv2.findTransformECC(
            ref_gray,
            tst_gray_ecc,
            warp_matrix,
            cv2.MOTION_EUCLIDEAN,
            crit,
            None,  # inputMask
            5      # gaussFiltSize
        )
    except cv2.error as e:
        logger.warning(f'ECC failed to converge: {e}')
        return _no_registration_result('ECC failed - returning tst_im unchanged')

    rot_deg = np.degrees(np.arctan2(warp_matrix[1, 0], warp_matrix[0, 0]))
    translation_xy = (warp_matrix[0, 2], warp_matrix[1, 2])
    logger.info(f'ECC correlation: {cc:.6f}')
    logger.info(f'Rotation: {rot_deg:.2f} degrees')
    logger.info(f'Translation dx: {translation_xy[0]:.3f}px, dy: {translation_xy[1]:.3f}px')

    if abs(rot_deg) > angle_limit:
        return _no_registration_result(
            f'Rotation {rot_deg:.2f} exceeds limit {angle_limit} - returning tst_im unchanged'
        )

    # Apply the transformation matrix to the test image
    h, w = ref_gray.shape[:2]
    rg_tst = cv2.warpAffine(
        tst_im,
        warp_matrix,
        (w, h),
        flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP,
        borderMode=cv2.BORDER_REPLICATE
    )

    ncc = _compute_ncc(ref_gray, _to_gray(rg_tst))
    logger.info(f'NCC: {ncc:.4f} (1.0=perfect match, 0.0=no correlation)')
    return ETImageRegistration(
        registered_test_et=rg_tst,
        rotation_deg=rot_deg,
        translation_xy=translation_xy,
        transformation_matrix=warp_matrix,
        dice_after=ncc,
        registration_ok=ncc > 0.9
    )


In [ ]:
rs = register_image(
    ref_im = reference_image,
    tst_im = tst_im,
    angle_limit = 6.0,
    upsample_factor = 20
    )

In [ ]:
show_(rs.registered_test_et)

---
#### Extract tiles

In [ ]:
#| export
def extract_tile(
    img: np.ndarray, # image
    y: int, # starting row position
    x: int, # starting column position
    tile_h: int, # tile height
    tile_w: int # tile width
) -> np.ndarray:
    """Extract a single tile from image."""
    return img[y:y+tile_h, x:x+tile_w].copy()

In [ ]:
#| export
def get_tile_positions(
    img_h: int, # Image height
    img_w: int, # Image width
    tile_h: int, # Tile height
    tile_w: int, # Tile width
    stride_h: int = None, # Vertical stride (default: tile_h, no overlap)
    stride_w: int = None, # Horizontal stride (default: tile_w, no overlap)
    include_edges: bool = True # If True, add edge tiles to cover full image
) -> List[Tuple[int, int]]:
    """Calculate starting positions for all tiles.
    
        
        
    Example:
        >>> positions = get_tile_positions(512, 512, 256, 256, stride_h=128)
        >>> len(positions)  # 9 tiles with 50% overlap
    """
    stride_h = stride_h or tile_h
    stride_w = stride_w or tile_w
    
    positions = []
    for y in range(0, img_h - tile_h + 1, stride_h):
        for x in range(0, img_w - tile_w + 1, stride_w):
            positions.append((y, x))
    
    # Add edge tiles if image isn't perfectly divisible
    if include_edges:
        # Right edge
        if (img_w - tile_w) % stride_w != 0:
            for y in range(0, img_h - tile_h + 1, stride_h):
                positions.append((y, img_w - tile_w))
        # Bottom edge
        if (img_h - tile_h) % stride_h != 0:
            for x in range(0, img_w - tile_w + 1, stride_w):
                positions.append((img_h - tile_h, x))
        # Bottom-right corner
        if (img_h - tile_h) % stride_h != 0 and (img_w - tile_w) % stride_w != 0:
            positions.append((img_h - tile_h, img_w - tile_w))
    
    # Remove duplicates while preserving order
    seen = set()
    unique_positions = []
    for pos in positions:
        if pos not in seen:
            seen.add(pos)
            unique_positions.append(pos)
    
    return unique_positions

In [ ]:
#| export
def extract_tiles(
    img: np.ndarray,
    tile_h: int,
    tile_w: int,
    stride_h: int = None,
    stride_w: int = None,
    include_edges: bool = True
) -> Tuple[List[np.ndarray], List[Tuple[int, int]]]:
    """Extract all tiles from an image.
    
    Args:
        img: Input image [H, W] or [H, W, C]
        tile_h: Tile height
        tile_w: Tile width
        stride_h: Vertical stride
        stride_w: Horizontal stride
        include_edges: Include edge tiles
        
    Returns:
        tiles: List of tile arrays
        positions: List of (y, x) positions for each tile
        
    Example:
        >>> tiles, positions = extract_tiles(img, 256, 256, stride_h=128)
    """
    img_h, img_w = img.shape[:2]
    positions = get_tile_positions(
        img_h, img_w, tile_h, tile_w, 
        stride_h, stride_w, include_edges
    )
    
    tiles = [extract_tile(img, y, x, tile_h, tile_w) for y, x in positions]
    return tiles, positions

In [ ]:
tiles, positions = extract_tiles(
    tst_im, 
    tile_h=256, 
    tile_w=256, 
    stride_h=None,
    stride_w=None,
    include_edges=True
    )
tiles[0].shape


In [ ]:
tile_no =[1, 4,6,7,10,11,13,16,17,19,20,21,22,23,29,30,31,32,33]
im_list = [
    tiles[1], 
    tiles[4], 
    tiles[5],
    tiles[7],
    tiles[10],
    tiles[11],
    tiles[13],
    tiles[16],
    tiles[17],
    tiles[19],
    tiles[20],
    tiles[21],
    tiles[22],
    tiles[23],
    tiles[29],
    tiles[30],
    tiles[31],
    tiles[32],
    tiles[33],
    ]
cols = len(im_list)
display_imgs(
    images=im_list,
    scale=1, 
    COLS=cols, 
    desc=None);

In [ ]:
display_imgs(
    images=tiles, 
    scale=1, 
    COLS=6, 
    desc=None);

In [ ]:
tst_im = cv2.imread(sm_1c_150, cv2.IMREAD_GRAYSCALE)


In [ ]:
#| export
def _to_display_img(img):
    """Convert img (numpy or torch.Tensor) to (H,W) uint8 for imshow."""
    if hasattr(img, 'detach'):  # PyTorch tensor
        arr = img.detach().cpu().numpy()
    else:
        arr = np.asarray(img, dtype=np.float32)
    # Handle 1D (e.g. phase_shift [dx,dy]) -> reshape to valid 2D for imshow
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)  # (2,) -> (1,2) so imshow accepts it
    # Handle (B,C,H,W) or (C,H,W) -> take first, squeeze channels
    if arr.ndim == 4:
        arr = arr[0]
    if arr.ndim == 3:
        arr = arr.squeeze()
        if arr.ndim == 3 and arr.shape[0] in (1, 3):  # (C,H,W)
            arr = arr.transpose(1, 2, 0).squeeze()
    # Scale to 0-255 for display (handles arbitrary ranges)
    if arr.size > 0 and arr.max() > arr.min():
        arr = (arr - arr.min()) / (arr.max() - arr.min()) * 255
    else:
        arr = np.zeros_like(arr) if arr.size > 0 else np.array([[0]])
    return np.clip(arr, 0, 255).astype(np.uint8)


def display_imgs(
    images, # images
    scale=2, # scale
    COLS=5, # columns
    desc=None # description
    ):
    """Display images in a grid. Accepts numpy arrays or PyTorch tensors (B,C,H,W or C,H,W)."""
    DPI = float(mpl.rcParams['figure.dpi'])
    ROWS = math.ceil(len(images) / COLS)
    
    # Get h,w from first image (handle tensor layout)
    img0 = images[0]
    if hasattr(img0, 'detach'):
        shp = img0.detach().cpu().numpy().shape
    else:
        shp = np.asarray(img0).shape
    if len(shp) >= 3 and shp[0] in (1, 3):  # (C,H,W)
        h, w = shp[1], shp[2]
    else:
        h, w = shp[0], shp[1]
    
    fig, axs = plt.subplots(ROWS, COLS, figsize=((COLS * w * scale) / DPI, (ROWS * h * scale) / DPI))
    
    if desc is not None:
        fig.suptitle(desc, fontsize=10)
        
    axs = np.array([axs]).reshape(-1, COLS)
    for r in range(ROWS):
        for c in range(COLS):
            pos = (r * COLS) + c
            if pos < len(images):
                img = _to_display_img(images[pos])
                axs[r][c].imshow(img, cmap="gray", vmin=0, vmax=255)
                
            axs[r][c].xaxis.set_major_locator(ticker.NullLocator())
            axs[r][c].yaxis.set_major_locator(ticker.NullLocator())        
    fig.subplots_adjust(wspace=0.01, hspace=0.01)
    fig.tight_layout()
    return fig

In [ ]:
ph_rs, phase_shift = phase_correlation_alignment(
    reference_img=reference_image,
    moving_img=tst_im,
    )
# Show moving (before) and aligned (after) side by side; phase_shift is [dx,dy] not an image
display_imgs(images=[reference_image,tst_im, ph_rs], scale=1, COLS=3, desc=None);

In [ ]:
alig_rs, homography = feature_based_alignment(
    reference_img=reference_image,
    moving_img=tst_im,
    detector_type='sift',
    )


In [ ]:
c_ref = center_crop(Image.fromarray(reference_image))
c_tst = center_crop(Image.fromarray(tst_im))
display_imgs(images=[c_ref,c_tst], scale=1, COLS=2, desc=None);

In [ ]:
align_rs, homography = feature_based_alignment(
    reference_img=np.array(c_ref),
    moving_img=np.array(c_tst),
    detector_type='sift',
    )
display_imgs(images=[np.array(c_ref),align_rs], scale=1, COLS=2, desc=None);

In [ ]:
rs = multi_method_alignment(
    reference_img=reference_image,
    moving_img=tst_im,
    methods = ['phase_correlation','feature_based', 'subpixel', 'optical_flow']
    )

In [ ]:
img_ = read_img(im_fn)

In [ ]:
show_(img_)

In [ ]:
nm = Path(im_fn).name
fn_ = fr'M:\Fail_investigation\Missing_lead\ETPD_WAR_1_02_missing_20240813\ETPD_WAR_1_02_missing_20240813_unzip\main_im2_cropped_masks\threshold52\failed\missing\missing_pin_sn_images\{nm}' 

In [ ]:
fn_=Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/ETPD_WAR_1_02_missing_20240813_unzip/main_im2_cropped_masks/threshold52/failed/missing/missing_pin_sn_images/tst.png')

In [ ]:
dst_path = Path('/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/ETPD_WAR_1_02_missing_20240813_unzip/main_im2_cropped_masks/threshold52_new/failed/missing/missing_pin_sn_images')

In [ ]:
dst_path.ls()

In [ ]:
show_(img_[340:440, 1070:1168])

In [ ]:
cv2.imwrite(
    f'{dst_path}/tst.png',
    img_[340:440, 1070:1168]
    )

In [ ]:
import shutil
import os

In [ ]:
path =Path(r'M:\Fail_investigation\Missing_lead\ETPD_WAR_1_02_missing_20240813\ETPD_WAR_1_02_missing_20240813_unzip\main_im2_cropped_masks\threshold52\failed\missing\missing_pin_sn_images')
path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/ETPD_WAR_1_02_missing_20240813_unzip/main_im2_cropped_masks/threshold52_new/failed/missing/missing_pin_sn_images')

In [ ]:
path.ls()

In [ ]:

for idx, i in enumerate(path.ls()):
    name_ = i.name
    pat = 'In_125_.*.png'
    pat = re.compile(pat)
    pa_ = re.findall(pat, name_)
    if pa_:
        nm = pa_[0]
        new_name = f'rec_{idx}_{nm}'
        Path(i).rename(Path(path,new_name))

In [ ]:
ffn = Path(fn_)

In [ ]:
show_(img_)

In [ ]:
os.getenv('PATH')
custom_lib_path = Path("/home/ai_warstein/homes/goni/custom_libs")
sys.path.append(str(custom_lib_path))
from dotenv import load_dotenv
load_dotenv(dotenv_path="/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/private_easy_pin_detection/.env")

In [ ]:
CV_TOOLS = os.getenv("CV_TOOLS")
P_EPD= os.getenv("PRIVATE_EASY_PIN_DETECTION")
sys.path.append(CV_TOOLS)

In [ ]:
from cv_tools.core import *

In [ ]:
im_path = Path('/home/ai_easypid.work/Reference_images')
ref_im2b = Path(im_path, 'ref_image_2B.png')

In [ ]:
img = read_img(ref_im2b)

In [ ]:
n_img = rotate_image(img, 7)
show_(n_img)

In [ ]:
sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(img, None)
kp2, des2 = sift.detectAndCompute(n_img, None)

In [ ]:
# FlANN parameters
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)
matches = flann.knnMatch(des1, des2, k=2)
good_matches = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good_matches.append(m)

In [ ]:
calculate_rotation(kp1, kp2, good_matches)

In [ ]:
def detect_features(
    img1: np.ndarray,
    img2: np.ndarray,
    method:str, # whether to use SIFT or ORB algorithm
    ) -> tuple[cv2.KeyPoint, cv2.KeyPoint, cv2.DMatch]:
    ' find best feature'

    if method == 'SIFT':
        detector = cv2.SIFT_create()
        kp1, des1 = detector.detectAndCompute(img1, None)
        kp2, des2 = detector.detectAndCompute(img2, None)

        FLANN_INDEX_KDTREE = 1
        index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
        search_params = dict(checks=50)
        flann = cv2.FlannBasedMatcher(
            index_params,
            search_params

            )
        matches = flann.knnMatch(des1, des2, k=2)
        good_matches =[] 
        for m,n in matches:
            if m.distance < 0.7 * n.distance:
                good_matches.append(m)
        return kp1, kp2, good_matches
    elif method == 'ORB':
        detector = cv2.ORB_create()
        kp1, des1 = detector.detectAndCompute(img1, None)
        kp2, des2 = detector.detectAndCompute(img2, None)
        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = bf.match(des1, des2)
        matches = sorted(matches, key = lambda x:x.distance)
        good_matches = matches[:10]
        return kp1, kp2, good_matches

In [ ]:
def calculate_rotation(
    kp1:cv2.KeyPoint, # keypoint of image 1 after feature detection
    kp2:cv2.KeyPoint, # keypoint of image 2 after feature detection
    good_matches : cv2.DMatch # good matches between the two images
    )-> float: # Angle of the rotation between the two images
    src_pts = np.float32(
        [kp1[m.queryIdx].pt for m in good_matches]
        ).reshape(-1, 1, 2)
    dst_pts = np.float32(
        [kp2[m.queryIdx].pt for m in good_matches]
    ).reshape(-1, 1, 2)
    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    angle = np.arctan2(M[1, 0], M[0, 0]) * 180 / np.pi

     
    # Normalize the angle to be between -180 and 180
    angle = (angle + 180) % 360 - 180

    if angle < -90:
        angle += 360
    return angle

In [ ]:
def calculate_rotationv02(
    kp1:cv2.KeyPoint, # keypoint of image 1 after feature detection
    kp2:cv2.KeyPoint, # keypoint of image 2 after feature detection
    good_matches : cv2.DMatch # good matches between the two images
    )-> float: # Angle of the rotation between the two images
    src_pts = np.float32(
        [kp1[m.queryIdx].pt for m in good_matches]
        ).reshape(-1,  2)
    dst_pts = np.float32(
        [kp2[m.queryIdx].pt for m in good_matches]
    ).reshape(-1,  2)


    # Calculate the center of mass of the points
    src_center = np.mean(src_pts, axis=0)
    dst_center = np.mean(dst_pts, axis=0)

    # Substruct the center to align the points
    src_centered = src_pts - src_center
    dst_centered = dst_pts - dst_center

    # Calculate the rotation angle
    angles = np.arctan2(
        dst_centered[:, 1], 
        dst_centered[:, 0]
        ) - np.arctan2(src_centered[:, 1], src_centered[:, 0])
    
    # Normalize angles to [-pi, pi]
    angles = (angles + np.pi) % (2 * np.pi) - np.pi

    # Calculate the median angle
    rotation_angle = np.median(angles) * 180 / np.pi

    return rotation_angle




In [ ]:
rows, cols = img.shape[:2]
center = (cols/2, rows/2)
rot_mat = cv2.getRotationMatrix2D(center, 0, 1.0)

In [ ]:
rotated = cv2.warpAffine(img, rot_mat, (cols, rows))

In [ ]:
rot_mat

In [ ]:
rotated.shape

In [ ]:
img.shape

In [ ]:
cv2.findTransformECC?

In [ ]:
cv2.findTransformECC(img, rotated, rot_mat, cv2.MOTION_EUCLIDEAN)

In [ ]:
def calculate_rotationv03(
    image1:np.ndarray,
    image2:np.ndarray,
    )-> float: # Angle of the rotation between the two images

    rows, cols = image1.shape[:2]
    center = (cols/2, rows/2)

    rot_mat = cv2.getRotationMatrix2D(center, 0, 1.0)
    rotated = cv2.warpAffine(image2, rot_mat, (cols, rows))

    h = cv2.findTransformECC(
        image1, 
        rotated, 
        rot_mat, 
        cv2.MOTION_EUCLIDEAN, 
        )[0]
    angle = np.arctan2(h[1, 0], h[0, 0]) * 180 / np.pi
    return angle
ang_ = calculate_rotationv03(img, n_img)

In [ ]:
show_(img)

In [ ]:
show_(n_img)

In [ ]:
180-179

In [ ]:
key1, key2, good_matches_f = detect_features(img, n_img, 'SIFT')
rot_angle_ = calculate_rotationv02(key1, key2, good_matches_f)
print(f' Rotation angle is {rot_angle_} degrees')
rotated_img = rotate_image(n_img, (rot_angle_))
show_(rotated_img)

In [ ]:
key1, key2, good_matches_f = detect_features(img, n_img, 'ORB')
rot_angle_ = calculate_rotation(key1, key2, good_matches_f)
print(f' Rotation angle is {rot_angle_} degrees')
rotated_img = rotate_image(n_img, (rot_angle_))
show_(rotated_img)

In [ ]:
show_(img)

In [ ]:
key1, key2, good_matches_f = detect_features(img, n_img, 'SIFT')
rot_angle_ = calculate_rotation(key1, key2, good_matches_f)
print(f' Rotation angle is {rot_angle_} degrees')
rotated_img = rotate_image(n_img, rot_angle_)
show_(rotated_img)

In [ ]:
show_(img)

In [ ]:
ang_ = calculate_rotation(kp1, kp2, good_matches)

In [ ]:
rot_img = rotate_image(n_img,ang_ )

In [ ]:
show_(rot_img)

In [ ]:
orb = cv2.ORB_create(50)
kp1, des1 = orb.detectAndCompute(img, None)
kp2, des2 = orb.detectAndCompute(n_img, None)
matcher = cv2.DescriptorMatcher_create(cv2.DESCRIPTOR_MATCHER_BRUTEFORCE_HAMMING)
matches = matcher.match(des1, des2, None)

im3 = cv2.drawMatches(
    img, 
    kp1, 
    n_img, 
    kp2, 
    matches[:10], 
    None, 
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

In [ ]:
show_(im3)

In [ ]:
pt1 = np.zeros(
    (len(matches), 2), dtype=np.float32) 
pt2 = np.zeros(
    (len(matches), 2), dtype=np.float32
    )

In [ ]:
for i, matches in enumerate(matches):
    pt1[i, :] = kp1[matches.queryIdx].pt
    pt2[i, :] = kp2[matches.trainIdx].pt

In [ ]:
h, mask = cv2.findHomography(pt1, pt2, cv2.RANSAC)

In [ ]:
new_img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)

In [ ]:
show_(img)

In [ ]:
show_(new_img)

In [ ]:
import cv2
import numpy as np

def detect_circle(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                               param1=50, param2=30, minRadius=10, maxRadius=100)
    
    if circles is not None:
        circles = np.round(circles[0, :]).astype("int")
        x, y, r = circles[0]
        return (x, y)
    return None

def calculate_angle(point1, point2):
    dx = point2[0] - point1[0]
    dy = point2[1] - point1[1]
    return np.degrees(np.arctan2(dy, dx))

def rotate_image(image, angle):
    center = tuple(np.array(image.shape[1::-1]) / 2)
    rot_mat = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, rot_mat, image.shape[1::-1], flags=cv2.INTER_LINEAR)

# Load reference and test images
reference_image = cv2.imread('reference_image.jpg')
test_image = cv2.imread('test_image.jpg')

# Detect circles in both images
ref_center = detect_circle(reference_image)
test_center = detect_circle(test_image)

if ref_center is None or test_center is None:
    print("Could not detect circles in one or both images.")
else:
    # Calculate the angle difference
    ref_angle = calculate_angle(ref_center, (ref_center[0], 0))
    test_angle = calculate_angle(test_center, (test_center[0], 0))
    rotation_angle = ref_angle - test_angle

    # Rotate the test image
    rotated_test_image = rotate_image(test_image, rotation_angle)

    # Display results
    cv2.imshow('Reference Image', reference_image)
    cv2.imshow('Original Test Image', test_image)
    cv2.imshow('Rotated Test Image', rotated_test_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    # Save the rotated test image
    cv2.imwrite('rotated_test_image.jpg', rotated_test_image)